# HiSparse 入门 (2/8)：三级存储池 —— Host / Resident / Hot

> **一句话结论**：历史 KV 分三级存放，越往设备侧越快、越小；三级之间用统一的 **656B 行**和 **int32 物理行号**寻址，`-1` 表示空行。

## 1. 三级池各自装什么

| 层级 | 位置 | 职责 | 比喻 |
| --- | --- | --- | --- |
| Host 池 | CPU 锁页内存（每 rank 私有，ACL AIV 注册） | 保存完整历史，容量最大，按需上行 | 图书馆仓库 |
| Resident 池 | 设备内存 | 每个请求已驻留的历史页表，常驻本请求 | 个人书架 |
| Hot 池 | 设备热区（slab） | 本步 top-k 命中行 + 热区槽表（dgi）+ LRU 顺序表 | 书桌桌面 |

## 2. 两个必须记住的不变量

### 2.1 统一行长：656B

无论数据在哪一级，一行的字节布局完全相同（下一篇细拆）。搬运时按整行拷贝，不需要按字段重组。

### 2.2 物理行号即地址

注意力拿到的是 int32 物理行号，直接换算地址：

```
地址 = pool_base + 行号 × 656
```

没有中间查表。行号为 `-1` 表示这一行是空行（掩码），注意力要跳过它。

## 3. 行号寻址示意（纯计算，不依赖 NPU）


In [ ]:
ROW_BYTES = 656
pool_base = 0  # 示意：实际是设备池起始地址

def row_address(row_idx):
    if row_idx == -1:
        return None  # 空行掩码
    return pool_base + row_idx * ROW_BYTES

for r in [0, 7, 128, -1]:
    print(f"row {r:>4} -> {row_address(r)}")


## 4. 三级之间怎么流动

1. 新请求的历史先在 Host 池留底；
2. resolver 判断哪些行本步需要，缺失的从 Host 搬到 Resident；
3. 本步 top-k 命中的行进入 Hot 池供注意力直读；
4. 用完按 LRU 顺序淘汰，被覆盖的旧行在索引里失效。

## 5. 小结

- 三级池：Host（完整历史）→ Resident（请求常驻）→ Hot（本步命中）；
- 统一 656B 行、物理行号直读、`-1` 空行掩码是跨层级的契约；
- 下一篇讲 resolver：谁来决定"哪本书上桌面"。

**下一篇**：`03-resolver-on-a5.ipynb`
